In [1]:
# Import required libraries
import sys
import os

# Add project root to path (adjust if needed)
sys.path.append('../')  # If notebook is in playground/ folder

print(f"📁 Current working directory: {os.getcwd()}")
print(f"🐍 Python path: {sys.path[-3:]}")  # Show last 3 entries

try:
    import pandas as pd
    import numpy as np
    print("✅ Basic libraries imported successfully!")
except ImportError as e:
    print(f"❌ Error importing basic libraries: {e}")

try:
    from rdkit import Chem
    from rdkit.Chem import AllChem, Draw
    print("✅ RDKit imported successfully!")
except ImportError as e:
    print(f"❌ RDKit not available: {e}")
    print("💡 Try: conda install -c conda-forge rdkit")

try:
    import torch
    from torch_geometric.data import Data, Dataset
    from torch_geometric.loader import DataLoader
    print(f"✅ PyTorch {torch.__version__} imported successfully!")
except ImportError as e:
    print(f"❌ PyTorch/PyG not available: {e}")
    print("💡 Try: pip install torch torch-geometric")

try:
    from IPython.display import display, HTML
    import matplotlib.pyplot as plt
    print("✅ Display libraries imported successfully!")
except ImportError as e:
    print(f"❌ Display libraries not available: {e}")

# Test scipy installation
try:
    import scipy
    print(f"✅ Scipy {scipy.__version__} imported successfully!")
except ImportError as e:
    print(f"❌ Scipy not available: {e}")

# Try normal MolE import first, fallback to workaround if needed
try:
    # Normal import - this should work now with scipy installed
    from mole.data.datasets import MolDataset, getAtomEnvironments, open_dictionary
    print("✅ MolE components imported successfully!")
    
except Exception as e:
    print(f"⚠️ Normal MolE import failed: {e}")
    print("🔄 Trying workaround method...")
    
    try:
        # Fallback: Direct import from specific files
        import importlib.util
        
        # Import datasets module directly
        datasets_path = "../mole/data/datasets.py"
        spec = importlib.util.spec_from_file_location("datasets", datasets_path)
        datasets_module = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(datasets_module)
        
        # Import vocabulary module directly  
        vocab_path = "../mole/data/vocabulary.py"
        spec = importlib.util.spec_from_file_location("vocabulary", vocab_path)
        vocab_module = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(vocab_module)
        
        # Extract the functions we need
        MolDataset = datasets_module.MolDataset
        getAtomEnvironments = datasets_module.getAtomEnvironments
        open_dictionary = vocab_module.open_dictionary
        
        print("✅ MolE components imported successfully (workaround)!")
        
    except Exception as e2:
        print(f"❌ Both normal and workaround imports failed: {e2}")
        print("💡 Please check your environment setup")

print("\n🎯 Import Summary Complete!")


📁 Current working directory: /home/mschauperl/programs/mole_public/playground
🐍 Python path: ['/home/mschauperl/anaconda3/envs/mole-env/lib/python3.10/site-packages', '/home/mschauperl/anaconda3/envs/mole-env/lib/python3.10/site-packages/setuptools/_vendor', '../']
✅ Basic libraries imported successfully!
✅ RDKit imported successfully!
✅ PyTorch 2.0.1+cu117 imported successfully!
✅ Display libraries imported successfully!
✅ Scipy 1.10.1 imported successfully!
⚠️ Normal MolE import failed: Unable to compare versions for numpy>=1.17: need=1.17 found=None. This is unusual. Consider reinstalling numpy.
🔄 Trying workaround method...
✅ MolE components imported successfully (workaround)!

🎯 Import Summary Complete!


In [17]:
from mole.data.datasets import MolDataset, getAtomEnvironments, open_dictionary


Unexpected exception formatting exception. Falling back to standard exception


Traceback (most recent call last):
  File "/home/mschauperl/.local/lib/python3.10/site-packages/IPython/core/interactiveshell.py", line 3398, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_165602/3652230886.py", line 1, in <cell line: 1>
    from mole.data.datasets import MolDataset, getAtomEnvironments, open_dictionary
  File "/home/mschauperl/programs/mole_public/playground/../mole/__init__.py", line 1, in <module>
    from .cli import mole_predict
  File "/home/mschauperl/programs/mole_public/playground/../mole/cli/mole_predict.py", line 11, in <module>
    import pytorch_lightning as pl
  File "/home/mschauperl/anaconda3/envs/mole-env/lib/python3.10/site-packages/pytorch_lightning/__init__.py", line 27, in <module>
    from pytorch_lightning.callbacks import Callback  # noqa: E402
  File "/home/mschauperl/anaconda3/envs/mole-env/lib/python3.10/site-packages/pytorch_lightning/callbacks/__init__.py", line 14, in <module>
    from pytorch_ligh

In [4]:
# Load the MolE vocabulary
vocab_path = "../mole/data/vocabularies/vocabulary_207atomenvs_radius0_ZINC_guacamole.pkl"
dictionary = open_dictionary(vocab_path)

print(f"📚 Vocabulary loaded successfully!")
print(f"   • Vocabulary size: {len(dictionary)}")
print(f"   • Special tokens: {[k for k in dictionary.keys() if isinstance(k, str)]}")

# Define our example molecules
molecules = {
    "ethanol": "CCO",
    "caffeine": "CN1C=NC2=C1C(=O)N(C(=O)N2C)C"
}

print(f"\n🧪 Example molecules:")
for name, smiles in molecules.items():
    mol = Chem.MolFromSmiles(smiles)
    print(f"   • {name.capitalize()}: {smiles} ({mol.GetNumAtoms()} atoms)")

# Create pandas Series (required input format for MolDataset)
smiles_series = pd.Series(list(molecules.values()))


📚 Vocabulary loaded successfully!
   • Vocabulary size: 211
   • Special tokens: ['PAD', 'MASK', 'UNK', 'CLS']

🧪 Example molecules:
   • Ethanol: CCO (3 atoms)
   • Caffeine: CN1C=NC2=C1C(=O)N(C(=O)N2C)C (14 atoms)


In [5]:
# Configuration 1: Basic dataset (no CLS token, structural environments)
dataset_basic = MolDataset(
    smiles=smiles_series,
    dictionary_inp=dictionary,
    radius_inp=0,           # Radius 0 atom environments
    labels=None,            # No labels (unsupervised)
    cls_token=False,        # No CLS token
    useFeatures_inp=False,  # Structural (not functional) environments
    use_class_weights=False
)

# Configuration 2: Supervised dataset with CLS token
dataset_supervised = MolDataset(
    smiles=smiles_series,
    dictionary_inp=dictionary,
    radius_inp=0,
    labels=np.array([[1.5, 0.8], [2.3, 1.2]]),  # Dummy labels (2 properties)
    cls_token=True,         # Add CLS token for classification
    useFeatures_inp=False,
    use_class_weights=False
)

# Configuration 3: Functional environments
dataset_functional = MolDataset(
    smiles=smiles_series,
    dictionary_inp=dictionary,
    radius_inp=0,
    labels=None,
    cls_token=False,
    useFeatures_inp=True,   # Use functional atom environments
    use_class_weights=False
)

print(f"📦 Created {len([dataset_basic, dataset_supervised, dataset_functional])} dataset configurations:")
print(f"   • Basic: {len(dataset_basic)} molecules")
print(f"   • Supervised: {len(dataset_supervised)} molecules") 
print(f"   • Functional: {len(dataset_functional)} molecules")


📦 Created 3 dataset configurations:
   • Basic: 2 molecules
   • Supervised: 2 molecules
   • Functional: 2 molecules


In [6]:
def analyze_molecule_processing(smiles, dictionary, molecule_name="molecule"):
    """Analyze how a molecule is processed by MolDataset"""
    
    print(f"🔬 Analysis of {molecule_name.capitalize()} ({smiles})")
    print("=" * 60)
    
    # Step 1: SMILES to RDKit molecule
    mol = Chem.MolFromSmiles(smiles)
    print(f"📊 Molecular Properties:")
    print(f"   • Number of atoms: {mol.GetNumAtoms()}")
    print(f"   • Number of bonds: {mol.GetNumBonds()}")
    print(f"   • Molecular formula: {Chem.rdMolDescriptors.CalcMolFormula(mol)}")
    
    # Step 2: Generate atom environments
    structural_envs = getAtomEnvironments(mol, dictionary, radius=0, useFeatures=False)
    functional_envs = getAtomEnvironments(mol, dictionary, radius=0, useFeatures=True)
    
    print(f"\n🎯 Atom Environment Tokenization:")
    print(f"   • Structural tokens: {structural_envs}")
    print(f"   • Functional tokens:  {functional_envs}")
    
    # Step 3: Analyze each atom
    print(f"\n🔍 Per-Atom Analysis:")
    for i, atom in enumerate(mol.GetAtoms()):
        element = atom.GetSymbol()
        degree = atom.GetDegree()
        hybridization = str(atom.GetHybridization())
        
        print(f"   Atom {i} ({element}):")
        print(f"     ├─ Degree: {degree}, Hybridization: {hybridization}")
        print(f"     ├─ Structural token: {structural_envs[i]}")
        print(f"     └─ Functional token:  {functional_envs[i]}")
    
    # Step 4: Distance matrix
    dist_mat = Chem.GetDistanceMatrix(mol)
    print(f"\n📏 Distance Matrix (Å):")
    print(f"   Shape: {dist_mat.shape}")
    print(f"   Matrix:")
    for i in range(dist_mat.shape[0]):
        row_str = "     " + " ".join([f"{dist_mat[i,j]:5.2f}" for j in range(dist_mat.shape[1])])
        print(row_str)
    
    return mol, structural_envs, functional_envs, dist_mat

# Analyze ethanol
ethanol_mol, eth_struct, eth_func, eth_dist = analyze_molecule_processing("CCO", dictionary, "ethanol")


🔬 Analysis of Ethanol (CCO)
📊 Molecular Properties:
   • Number of atoms: 3
   • Number of bonds: 2
   • Molecular formula: C2H6O

🎯 Atom Environment Tokenization:
   • Structural tokens: [13, 32, 124]
   • Functional tokens:  [209, 209, 209]

🔍 Per-Atom Analysis:
   Atom 0 (C):
     ├─ Degree: 1, Hybridization: SP3
     ├─ Structural token: 13
     └─ Functional token:  209
   Atom 1 (C):
     ├─ Degree: 2, Hybridization: SP3
     ├─ Structural token: 32
     └─ Functional token:  209
   Atom 2 (O):
     ├─ Degree: 1, Hybridization: SP3
     ├─ Structural token: 124
     └─ Functional token:  209

📏 Distance Matrix (Å):
   Shape: (3, 3)
   Matrix:
      0.00  1.00  2.00
      1.00  0.00  1.00
      2.00  1.00  0.00


In [7]:
def examine_dataset_output(dataset, molecule_idx, molecule_name, config_name):
    """Examine the complete output from MolDataset for a specific molecule"""
    
    print(f"📦 {config_name} Dataset Output for {molecule_name}")
    print("=" * 70)
    
    # Get the data object
    data = dataset[molecule_idx]
    
    print(f"🔍 PyTorch Geometric Data Object:")
    print(f"   • Type: {type(data)}")
    print(f"   • Keys: {list(data.keys)}")
    print()
    
    # Examine each component
    if hasattr(data, 'x'):
        print(f"📊 Node Features (x):")
        print(f"   • Shape: {data.x.shape}")
        print(f"   • Data type: {data.x.dtype}")
        print(f"   • Values: {data.x.tolist()}")
        print()
    
    if hasattr(data, 'edge_index'):
        print(f"🔗 Edge Index:")
        print(f"   • Shape: {data.edge_index.shape}")
        print(f"   • Data type: {data.edge_index.dtype}")
        print(f"   • Number of edges: {data.edge_index.shape[1]}")
        print(f"   • Source nodes: {data.edge_index[0].tolist()}")
        print(f"   • Target nodes: {data.edge_index[1].tolist()}")
        print()
    
    if hasattr(data, 'edge_attr'):
        print(f"⚖️ Edge Attributes (distances):")
        print(f"   • Shape: {data.edge_attr.shape}")
        print(f"   • Data type: {data.edge_attr.dtype}")
        print(f"   • Min distance: {data.edge_attr.min().item():.3f}")
        print(f"   • Max distance: {data.edge_attr.max().item():.3f}")
        print(f"   • Values: {data.edge_attr.tolist()}")
        print()
    
    if hasattr(data, 'target_labels'):
        print(f"🎯 Target Labels:")
        print(f"   • Shape: {data.target_labels.shape}")
        print(f"   • Values: {data.target_labels.tolist()}")
        print()
    
    return data

# Examine outputs for ethanol (index 0)
print("Examining different dataset configurations for ETHANOL:")
print()

data_basic = examine_dataset_output(dataset_basic, 0, "Ethanol", "Basic")
data_supervised = examine_dataset_output(dataset_supervised, 0, "Ethanol", "Supervised")
data_functional = examine_dataset_output(dataset_functional, 0, "Ethanol", "Functional")


Examining different dataset configurations for ETHANOL:

📦 Basic Dataset Output for Ethanol
🔍 PyTorch Geometric Data Object:
   • Type: <class 'torch_geometric.data.data.Data'>
   • Keys: ['edge_index', 'edge_attr', 'x']

📊 Node Features (x):
   • Shape: torch.Size([3])
   • Data type: torch.int64
   • Values: [13, 32, 124]

🔗 Edge Index:
   • Shape: torch.Size([2, 9])
   • Data type: torch.int64
   • Number of edges: 9
   • Source nodes: [0, 0, 0, 1, 1, 1, 2, 2, 2]
   • Target nodes: [0, 1, 2, 0, 1, 2, 0, 1, 2]

⚖️ Edge Attributes (distances):
   • Shape: torch.Size([9])
   • Data type: torch.int64
   • Min distance: 1.000
   • Max distance: 3.000
   • Values: [1, 2, 3, 2, 1, 2, 3, 2, 1]

📦 Supervised Dataset Output for Ethanol
🔍 PyTorch Geometric Data Object:
   • Type: <class 'torch_geometric.data.data.Data'>
   • Keys: ['edge_index', 'target_labels', 'edge_attr', 'x']

📊 Node Features (x):
   • Shape: torch.Size([4])
   • Data type: torch.int64
   • Values: [210, 13, 32, 124]

🔗 Ed

In [8]:
# Analyze caffeine processing
caffeine_mol, caff_struct, caff_func, caff_dist = analyze_molecule_processing("CN1C=NC2=C1C(=O)N(C(=O)N2C)C", dictionary, "caffeine")

print("\n" + "="*80)
print()

# Get caffeine data from supervised dataset
caffeine_data = examine_dataset_output(dataset_supervised, 1, "Caffeine", "Supervised")


🔬 Analysis of Caffeine (CN1C=NC2=C1C(=O)N(C(=O)N2C)C)
📊 Molecular Properties:
   • Number of atoms: 14
   • Number of bonds: 15
   • Molecular formula: C8H10N4O2

🎯 Atom Environment Tokenization:
   • Structural tokens: [13, 180, 144, 159, 150, 150, 150, 54, 180, 150, 54, 180, 13, 13]
   • Functional tokens:  [209, 209, 209, 209, 209, 209, 209, 209, 209, 209, 209, 209, 209, 209]

🔍 Per-Atom Analysis:
   Atom 0 (C):
     ├─ Degree: 1, Hybridization: SP3
     ├─ Structural token: 13
     └─ Functional token:  209
   Atom 1 (N):
     ├─ Degree: 3, Hybridization: SP2
     ├─ Structural token: 180
     └─ Functional token:  209
   Atom 2 (C):
     ├─ Degree: 2, Hybridization: SP2
     ├─ Structural token: 144
     └─ Functional token:  209
   Atom 3 (N):
     ├─ Degree: 2, Hybridization: SP2
     ├─ Structural token: 159
     └─ Functional token:  209
   Atom 4 (C):
     ├─ Degree: 3, Hybridization: SP2
     ├─ Structural token: 150
     └─ Functional token:  209
   Atom 5 (C):
     ├─ Degr

In [9]:
def compare_tokenization():
    """Compare tokenization across different configurations"""
    
    print("🔄 Tokenization Comparison")
    print("=" * 50)
    
    molecules_info = [
        ("Ethanol", "CCO", 0),
        ("Caffeine", "CN1C=NC2=C1C(=O)N(C(=O)N2C)C", 1)
    ]
    
    for name, smiles, idx in molecules_info:
        print(f"\n📊 {name} ({smiles}):")
        
        # Get data from different configurations
        basic_data = dataset_basic[idx]
        supervised_data = dataset_supervised[idx]
        functional_data = dataset_functional[idx]
        
        print(f"   🔹 Basic tokens:      {basic_data.x.flatten().tolist()}")
        print(f"   🔹 Supervised tokens: {supervised_data.x.flatten().tolist()}")
        print(f"   🔹 Functional tokens: {functional_data.x.flatten().tolist()}")
        
        # Check for CLS token
        if supervised_data.x.shape[0] > basic_data.x.shape[0]:
            cls_token_id = supervised_data.x[0].item()
            print(f"   🎯 CLS token added: {cls_token_id} (at position 0)")
        
        # Compare sequence lengths
        print(f"   📏 Sequence lengths:")
        print(f"      ├─ Basic: {basic_data.x.shape[0]} tokens")
        print(f"      ├─ Supervised: {supervised_data.x.shape[0]} tokens")
        print(f"      └─ Functional: {functional_data.x.shape[0]} tokens")

compare_tokenization()


🔄 Tokenization Comparison

📊 Ethanol (CCO):
   🔹 Basic tokens:      [13, 32, 124]
   🔹 Supervised tokens: [210, 13, 32, 124]
   🔹 Functional tokens: [209, 209, 209]
   🎯 CLS token added: 210 (at position 0)
   📏 Sequence lengths:
      ├─ Basic: 3 tokens
      ├─ Supervised: 4 tokens
      └─ Functional: 3 tokens

📊 Caffeine (CN1C=NC2=C1C(=O)N(C(=O)N2C)C):
   🔹 Basic tokens:      [13, 180, 144, 159, 150, 150, 150, 54, 180, 150, 54, 180, 13, 13]
   🔹 Supervised tokens: [210, 13, 180, 144, 159, 150, 150, 150, 54, 180, 150, 54, 180, 13, 13]
   🔹 Functional tokens: [209, 209, 209, 209, 209, 209, 209, 209, 209, 209, 209, 209, 209, 209]
   🎯 CLS token added: 210 (at position 0)
   📏 Sequence lengths:
      ├─ Basic: 14 tokens
      ├─ Supervised: 15 tokens
      └─ Functional: 14 tokens


In [14]:
def analyze_graph_structure(data, molecule_name):
    """Analyze the graph structure from MolDataset output"""

    print(f"🕸️ Graph Structure Analysis for {molecule_name}")
    print("=" * 60)

    # Edge connectivity
    edge_index = data.edge_index
    edge_attr = data.edge_attr
    num_nodes = data.x.shape[0]

    print(f"📊 Graph Statistics:")
    print(f"   • Number of nodes: {num_nodes}")
    print(f"   • Number of edges: {edge_index.shape[1]}")
    print(f"   • Is fully connected: {edge_index.shape[1] == num_nodes * num_nodes}")

    # Create adjacency matrix from edge data
    adj_matrix = torch.zeros(num_nodes, num_nodes)
    for i in range(edge_index.shape[1]):
        src, dst = edge_index[0, i].item(), edge_index[1, i].item()
        adj_matrix[src, dst] = edge_attr[i].item()

    print(f"\n📏 Distance/Adjacency Matrix:")
    print("     " + " ".join([f"{i:5d}" for i in range(num_nodes)]))
    for i in range(num_nodes):
        row_str = f"{i:2d}:  " + " ".join([f"{adj_matrix[i,j]:5.1f}" for j in range(num_nodes)])
        print(row_str)

    # Analyze distance distribution
    distances = edge_attr[edge_attr > 0]  # Exclude self-connections (0 distance)
    if len(distances) > 0:
        print(f"\n📈 Distance Statistics:")
        print(f"   • Min distance: {distances.min():.3f} Å")
        print(f"   • Max distance: {distances.max():.3f} Å")
        print(f"   • Mean distance: {np.array([float(x) for x in distances]).mean():.3f} Å")
        print(
            f"   • Std distance: {np.array([float(x) for x in distances]).std():.3f} Å"
        )

# Analyze graph structure for both molecules
analyze_graph_structure(data_basic, "Ethanol")
print("\n" + "="*80 + "\n")
analyze_graph_structure(caffeine_data, "Caffeine")

🕸️ Graph Structure Analysis for Ethanol
📊 Graph Statistics:
   • Number of nodes: 3
   • Number of edges: 9
   • Is fully connected: True

📏 Distance/Adjacency Matrix:
         0     1     2
 0:    1.0   2.0   3.0
 1:    2.0   1.0   2.0
 2:    3.0   2.0   1.0

📈 Distance Statistics:
   • Min distance: 1.000 Å
   • Max distance: 3.000 Å
   • Mean distance: 1.889 Å
   • Std distance: 0.737 Å


🕸️ Graph Structure Analysis for Caffeine
📊 Graph Statistics:
   • Number of nodes: 15
   • Number of edges: 196
   • Is fully connected: False

📏 Distance/Adjacency Matrix:
         0     1     2     3     4     5     6     7     8     9    10    11    12    13    14
 0:    0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0
 1:    0.0   1.0   2.0   3.0   4.0   4.0   3.0   4.0   5.0   5.0   6.0   7.0   5.0   6.0   6.0
 2:    0.0   2.0   1.0   2.0   3.0   3.0   2.0   3.0   4.0   4.0   5.0   6.0   4.0   5.0   5.0
 3:    0.0   3.0   2.0   1.0   2.0   3.0   3.0   4.0

In [15]:
def print_summary():
    """Print comprehensive summary of MolDataset functionality"""
    
    print("📋 MolDataset Processing Summary")
    print("=" * 80)
    
    print("""
🔄 PROCESSING PIPELINE:
   1. SMILES String → RDKit Molecule object
   2. Molecule → Morgan Fingerprints (atom environments)
   3. Environment Hashes → Token IDs (via vocabulary lookup)
   4. Distance Matrix → Sparse graph representation
   5. Combine into PyTorch Geometric Data object

🎯 KEY COMPONENTS:
   • Node Features (x): Token IDs representing atom environments
   • Edge Index: Fully connected graph (all atoms connected to all atoms)
   • Edge Attributes: 3D distances between atoms (in Angstroms)
   • Target Labels: Optional supervision signals for property prediction

⚙️ CONFIGURATION OPTIONS:
   • radius_inp: Environment radius (0=immediate, 1=neighbors, etc.)
   • useFeatures_inp: False=structural, True=functional environments
   • cls_token: Add classification token at sequence start
   • labels: Supervision targets for property prediction

🧠 MODEL-READY OUTPUT:
   • Compatible with PyTorch Geometric and standard PyTorch
   • Preserves both local (tokens) and global (distances) information
   • Handles variable molecule sizes via graph representation
   • Ready for transformer or graph neural network architectures
    """)
    
    # Show vocabulary statistics
    special_tokens = {k: v for k, v in dictionary.items() if isinstance(k, str)}
    env_tokens = {k: v for k, v in dictionary.items() if isinstance(k, int)}
    
    print(f"📚 VOCABULARY STATISTICS:")
    print(f"   • Total vocabulary size: {len(dictionary)}")
    print(f"   • Environment tokens: {len(env_tokens)}")
    print(f"   • Special tokens: {len(special_tokens)}")
    print(f"   • Special token mapping: {special_tokens}")
    
    print(f"\n🔬 MOLECULE EXAMPLES:")
    for i, (name, smiles) in enumerate(molecules.items()):
        basic_data = dataset_basic[i]
        supervised_data = dataset_supervised[i]
        
        print(f"   • {name.capitalize()} ({smiles}):")
        print(f"     ├─ Atoms: {len(basic_data.x)} tokens")
        print(f"     ├─ Edges: {basic_data.edge_index.shape[1]} connections")
        print(f"     └─ With CLS: {len(supervised_data.x)} tokens")

print_summary()


📋 MolDataset Processing Summary

🔄 PROCESSING PIPELINE:
   1. SMILES String → RDKit Molecule object
   2. Molecule → Morgan Fingerprints (atom environments)
   3. Environment Hashes → Token IDs (via vocabulary lookup)
   4. Distance Matrix → Sparse graph representation
   5. Combine into PyTorch Geometric Data object

🎯 KEY COMPONENTS:
   • Node Features (x): Token IDs representing atom environments
   • Edge Index: Fully connected graph (all atoms connected to all atoms)
   • Edge Attributes: 3D distances between atoms (in Angstroms)
   • Target Labels: Optional supervision signals for property prediction

⚙️ CONFIGURATION OPTIONS:
   • radius_inp: Environment radius (0=immediate, 1=neighbors, etc.)
   • useFeatures_inp: False=structural, True=functional environments
   • cls_token: Add classification token at sequence start
   • labels: Supervision targets for property prediction

🧠 MODEL-READY OUTPUT:
   • Compatible with PyTorch Geometric and standard PyTorch
   • Preserves both lo

In [16]:
from torch_geometric.loader import DataLoader

def demonstrate_dataloader_integration():
    """Show how MolDataset integrates with PyTorch data loaders"""
    
    print("🔄 DataLoader Integration Demo")
    print("=" * 50)
    
    # Create a DataLoader with our supervised dataset
    dataloader = DataLoader(
        dataset_supervised,
        batch_size=2,  # Both molecules in one batch
        shuffle=False,
        num_workers=0  # Avoid multiprocessing issues in notebook
    )
    
    print(f"📦 Created DataLoader:")
    print(f"   • Dataset size: {len(dataset_supervised)}")
    print(f"   • Batch size: {dataloader.batch_size}")
    print(f"   • Number of batches: {len(dataloader)}")
    
    # Iterate through batches
    for batch_idx, batch in enumerate(dataloader):
        print(f"\n🎯 Batch {batch_idx}:")
        print(f"   • Batch type: {type(batch)}")
        print(f"   • Node features shape: {batch.x.shape}")
        print(f"   • Edge index shape: {batch.edge_index.shape}")
        print(f"   • Edge attributes shape: {batch.edge_attr.shape}")
        print(f"   • Batch vector: {batch.batch.tolist()}")
        print(f"   • Target labels shape: {batch.target_labels.shape}")
        
        # Show how molecules are batched together
        print(f"   🔍 Molecule separation:")
        unique_batch_ids = torch.unique(batch.batch)
        for mol_id in unique_batch_ids:
            mask = batch.batch == mol_id
            mol_tokens = batch.x[mask]
            mol_name = list(molecules.keys())[mol_id]
            print(f"      ├─ Molecule {mol_id} ({mol_name}): {len(mol_tokens)} tokens")
            print(f"      └─ Tokens: {mol_tokens.flatten().tolist()}")

demonstrate_dataloader_integration()


🔄 DataLoader Integration Demo
📦 Created DataLoader:
   • Dataset size: 2
   • Batch size: 2
   • Number of batches: 1

🎯 Batch 0:
   • Batch type: <class 'torch_geometric.data.batch.DataBatch'>
   • Node features shape: torch.Size([19])
   • Edge index shape: torch.Size([2, 205])
   • Edge attributes shape: torch.Size([205])
   • Batch vector: [0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
   • Target labels shape: torch.Size([4])
   🔍 Molecule separation:
      ├─ Molecule 0 (ethanol): 4 tokens
      └─ Tokens: [210, 13, 32, 124]
      ├─ Molecule 1 (caffeine): 15 tokens
      └─ Tokens: [210, 13, 180, 144, 159, 150, 150, 150, 54, 180, 150, 54, 180, 13, 13]
